# Atelier 2 → Site web gratuit (Streamlit)

Ce notebook sert à entraîner **les mêmes six modèles** sur vos images DB2C et à créer le modèle à ajouter au site web. 

**Un seul entraînement nécessaire.** Ensuite le professeur pourra importer de nouvelles images directement sur le lien Streamlit, sans compte et sans entraînement.

Dossier attendu dans Google Drive : `Mon Drive / DB2C / obj_car` et `Mon Drive / DB2C / obj_ship`.


## Cellule 1 — Imports, connexion à Drive et calcul des 52 caractéristiques
Cette cellule conserve l’extraction RGB (6), HSV (24), GLCM (4), LBP (11), Hu (7).

In [ ]:
# -*- coding: utf-8 -*-
"""Atelier 2 — classification supervisée et test de nouvelles images.

Sur Google Colab :
  1. Conserver le jeu de données dans Mon Drive/DB2C/obj_car et obj_ship.
  2. Exécuter le programme (l'entraînement se fait sur le jeu de données).
  3. Choisir une ou plusieurs images sur l'ordinateur quand Colab le demande.

Dans le notebook .ipynb associé, la dernière cellule peut être relancée
indépendamment pour tester de nouvelles images SANS réentraîner les modèles.

IMPORTANT : les classes apprises sont uniquement Voiture et Bateau.
Une image d'une autre catégorie sera quand même affectée à l'une de ces
classes ; le résultat ne prouvera pas que l'objet est une voiture ou un bateau.
"""

import os
import time
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.color import rgb2hsv, rgb2gray
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

try:
    from google.colab import drive, files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    drive.mount('/content/drive')

DATASET_PATH = Path(os.environ.get(
    'ATELIER_DATASET_PATH',
    '/content/drive/MyDrive/DB2C' if EN_COLAB else 'DB2C'
))

CLASS_FOLDERS = {'obj_car': 0, 'obj_ship': 1}
CLASS_NAMES = {0: 'Voiture', 1: 'Bateau'}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}


def charger_image(chemin):
    """Lecture en RGB (OpenCV lit normalement en BGR)."""
    image_bgr = cv2.imread(str(chemin), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError(f'Impossible de lire cette image : {chemin}')
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def extract_one(image):
    """52 caractéristiques : RGB(6), HSV(24), GLCM(4), LBP(11), Hu(7)."""
    rgb = image.astype(np.float32) / 255.0

    # Couleur RGB : 3 moyennes + 3 écarts-types.
    rgb_stats = np.concatenate((rgb.mean(axis=(0, 1)), rgb.std(axis=(0, 1))))

    # Couleur HSV : 8 valeurs par histogramme, pour 3 canaux.
    hsv = rgb2hsv(rgb)
    hsv_hist = np.concatenate([
        np.histogram(hsv[:, :, channel], bins=8, range=(0, 1))[0]
        / (rgb.shape[0] * rgb.shape[1])
        for channel in range(3)
    ])

    # Texture : matrice de cooccurrence (GLCM).
    gray8 = np.round(rgb2gray(rgb) * 255).astype(np.uint8)
    quantized = (gray8 // 32).astype(np.uint8)
    glcm = graycomatrix(
        quantized, distances=[1, 2], angles=[0, np.pi / 2],
        levels=8, symmetric=True, normed=True
    )
    glcm_features = np.array([
        graycoprops(glcm, prop).mean()
        for prop in ('contrast', 'dissimilarity', 'homogeneity', 'energy')
    ])

    # Texture : motif binaire local (LBP).
    lbp = local_binary_pattern(gray8, P=8, R=1, method='uniform')
    lbp_hist = np.histogram(lbp, bins=np.arange(0, 12))[0].astype(np.float64)
    lbp_hist /= max(lbp_hist.sum(), 1)

    # Forme : 7 moments de Hu.
    hu = cv2.HuMoments(cv2.moments(gray8)).ravel()
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)

    features = np.concatenate((rgb_stats, hsv_hist, glcm_features, lbp_hist, hu))
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)


def extract_dataset(dataset, name):
    start = time.time()
    result = np.vstack([extract_one(img) for img in dataset]).astype(np.float32)
    print(f'{name} : {result.shape}, temps : {time.time() - start:.1f} s')
    return result


def adapter_taille(image, taille_reference):
    """Même format pour données d'entraînement et nouvelles images.

    Si les images de DB2C sont des CIFAR-10 32x32, taille_reference = (32, 32).
    """
    largeur, hauteur = taille_reference
    if image.shape[:2] == (hauteur, largeur):
        return image
    interpolation = cv2.INTER_AREA if (image.shape[1] > largeur or image.shape[0] > hauteur) else cv2.INTER_LINEAR
    return cv2.resize(image, (largeur, hauteur), interpolation=interpolation)




## Cellule 2 — Entraînement et résultats complets de l’atelier
Chargement des deux classes, séparation 80/20, SVM RBF, matrice de confusion, rapport de classification, accuracy et classement des six algorithmes.

In [ ]:
# A — Charger les deux classes du dataset (pas seulement deux photos).
if not DATASET_PATH.is_dir():
    raise FileNotFoundError(
        f'Dataset absent : {DATASET_PATH}\n'
        'Sur Colab, placer les dossiers obj_car et obj_ship dans Mon Drive/DB2C.'
    )

images, labels = [], []
for dossier, classe in CLASS_FOLDERS.items():
    chemin = DATASET_PATH / dossier
    if not chemin.is_dir():
        raise FileNotFoundError(f'Dossier de classe introuvable : {chemin}')

    fichiers = sorted(
        p for p in chemin.rglob('*')
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    compteur = 0
    for fichier in fichiers:
        try:
            image = charger_image(fichier)
        except ValueError:
            print('Image ignorée (illisible) :', fichier.name)
            continue
        images.append(image)
        labels.append(classe)
        compteur += 1
    print(f'{CLASS_NAMES[classe]} : {compteur} images')

if not images:
    raise ValueError('Aucune image lisible trouvée dans le dataset.')

labels = np.asarray(labels, dtype=np.int64)
for classe in CLASS_NAMES:
    if np.sum(labels == classe) < 5:
        raise ValueError(f'Prévoir au moins 5 images par classe ({CLASS_NAMES[classe]}).')

# La nouvelle image utilisera la même résolution que les données apprises.
TAILLE_REFERENCE = (images[0].shape[1], images[0].shape[0])  # largeur, hauteur
images = [adapter_taille(img, TAILLE_REFERENCE) for img in images]
print('Total :', len(images), '| Résolution utilisée :', TAILLE_REFERENCE)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, classe in zip(axes, [0, 1]):
    indice = int(np.flatnonzero(labels == classe)[0])
    ax.imshow(images[indice])
    ax.set_title(CLASS_NAMES[classe])
    ax.axis('off')
plt.suptitle('Exemples des classes du dataset')
plt.tight_layout()
plt.show()

indices_train, indices_test = train_test_split(
    np.arange(len(labels)), test_size=0.20, random_state=42, stratify=labels
)
x_train = [images[i] for i in indices_train]
y_train = labels[indices_train]
x_test = [images[i] for i in indices_test]
y_test = labels[indices_test]
print('Images apprentissage :', len(x_train), '| Images test :', len(x_test))

# B — Les mêmes 52 caractéristiques sont calculées pour chaque image.
X_train = extract_dataset(x_train, 'Apprentissage')
X_test = extract_dataset(x_test, 'Test')
assert X_train.shape[1] == X_test.shape[1] == 52
print('Caractéristiques par image :', X_train.shape[1])

# C — SVM RBF : entraînement et exemple issu du dataset de test.
rbf_model = make_pipeline(StandardScaler(), SVC(kernel='rbf'))
debut = time.time()
rbf_model.fit(X_train, y_train)
print(f'Temps entraînement SVM RBF : {time.time() - debut:.1f} s')

sample_index = 0
sample_prediction = int(rbf_model.predict(X_test[sample_index:sample_index + 1])[0])
sample_truth = int(y_test[sample_index])
plt.figure(figsize=(4, 4))
plt.imshow(x_test[sample_index])
plt.title(f'Vrai : {CLASS_NAMES[sample_truth]} | Prédit : {CLASS_NAMES[sample_prediction]}')
plt.axis('off')
plt.show()
print('Exemple de test — classe réelle :', CLASS_NAMES[sample_truth])
print('Exemple de test — classe prédite :', CLASS_NAMES[sample_prediction])

# D — Les métriques suivantes portent UNIQUEMENT sur les images étiquetées du test.
y_pred = rbf_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[CLASS_NAMES[i] for i in (0, 1)],
    yticklabels=[CLASS_NAMES[i] for i in (0, 1)]
)
plt.xlabel('Classe prédite')
plt.ylabel('Classe réelle')
plt.title('Matrice de confusion — SVM RBF')
plt.tight_layout()
plt.show()

print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=[CLASS_NAMES[i] for i in (0, 1)],
    digits=4, zero_division=0
))
print(f'Accuracy SVM RBF : {accuracy_score(y_test, y_pred):.4f}')

# E — Comparaison des SIX mêmes algorithmes du fichier original.
models = {
    'SVM RBF': rbf_model,
    'SVM linéaire': make_pipeline(StandardScaler(), SVC(kernel='linear')),
    'SVM sigmoïde': make_pipeline(StandardScaler(), SVC(kernel='sigmoid')),
    'KNN (k=3)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
    'KNN (k=5)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'Arbre de décision': DecisionTreeClassifier(random_state=42),
}

comparison = []
for name, model in models.items():
    debut = time.time()
    if name != 'SVM RBF':
        model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    score = accuracy_score(y_test, predictions)
    comparison.append((name, score))
    print(f'{name:20s} | Accuracy = {score:.4f} | Temps = {time.time() - debut:.1f} s')

comparison.sort(key=lambda element: element[1], reverse=True)
print('\nClassement des modèles :')
for rang, (nom, score) in enumerate(comparison, start=1):
    print(f'{rang}. {nom} : {score:.4f}')
print(f'\nMeilleur modèle : {comparison[0][0]} ({comparison[0][1]:.4f})')
print(f'Moins performant : {comparison[-1][0]} ({comparison[-1][1]:.4f})')

plt.figure(figsize=(9, 5))
sns.barplot(
    x=[score for _, score in comparison],
    y=[name for name, _ in comparison], color='steelblue'
)
plt.xlim(0, 1)
plt.xlabel('Accuracy sur les données de test')
plt.ylabel('Modèle')
plt.title('Comparaison des six algorithmes')
plt.tight_layout()
plt.show()




## Cellule 3 — Exporter les modèles pour créer votre site web

**Cette étape est obligatoire.** Exécutez la cellule ci-dessous après l'entraînement et téléchargez `export_site.zip`. Décompressez-le : vous obtiendrez `modele_atelier.joblib` (le modèle) et `requirements.txt` (les versions utilisées). Ajoutez-les dans le même dépôt GitHub que `streamlit_app.py` et `caracteristiques.py`.

Si Chrome bloque le téléchargement automatique, ouvrez le panneau *Fichiers* à gauche dans Colab et téléchargez `export_site.zip` manuellement.

In [ ]:
# Créer le fichier de modèles ET les résultats de l'atelier pour le site.
# Exécuter seulement APRÈS les cellules d'entraînement ci-dessus.
import zipfile
import joblib
import sklearn
import skimage
import matplotlib
import pandas

resultats_6 = {}
for nom, modele in models.items():
    preds = modele.predict(X_test)
    resultats_6[nom] = {
        'accuracy': float(accuracy_score(y_test, preds)),
        'confusion_matrix': confusion_matrix(y_test, preds, labels=[0, 1]).tolist(),
        'classification_report': classification_report(
            y_test, preds, labels=[0, 1],
            target_names=[CLASS_NAMES[0], CLASS_NAMES[1]],
            output_dict=True, zero_division=0
        ),
    }

artefact_site = {
    'version': 'atelier2-website-v1',
    'modeles': models,
    'classes': CLASS_NAMES,
    'taille_reference': TAILLE_REFERENCE,
    'evaluations': resultats_6,
    'nb_train': len(x_train),
    'nb_test': len(x_test),
    'nb_total': len(images),
    'versions': {'numpy': np.__version__, 'scikit-learn': sklearn.__version__,
                 'scikit-image': skimage.__version__, 'opencv': cv2.__version__},
}
joblib.dump(artefact_site, 'modele_atelier.joblib', compress=3)

# Aligner les versions scientifiques avec celles qui ont entraîné le modèle.
requirements_site = f"""streamlit>=1.44,<2
numpy=={np.__version__}
scikit-learn=={sklearn.__version__}
scikit-image=={skimage.__version__}
opencv-python-headless>={cv2.__version__},<5
matplotlib>=3.8,<4
pandas>=2.1,<3
joblib=={joblib.__version__}
"""
Path('requirements.txt').write_text(requirements_site, encoding='utf-8')

with zipfile.ZipFile('export_site.zip', 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write('modele_atelier.joblib')
    z.write('requirements.txt')

print('Fichier produit : export_site.zip')
print('Entraînement :', len(x_train), 'images | Test :', len(x_test), 'images')
print('Meilleur algorithme :', max(resultats_6, key=lambda n: resultats_6[n]['accuracy']))
print('Le ZIP contient le modèle réel et les versions de bibliothèques à publier sur GitHub.')
if EN_COLAB:
    from google.colab import files
    files.download('export_site.zip')


## Cellule 4 — Test facultatif d'une image avant la publication

Le professeur pourra ensuite faire la même chose directement depuis l'application Streamlit. Cette cellule n'est pas nécessaire pour exporter le modèle.

In [ ]:
# Sur Colab : une fenêtre demande de choisir la/les photo(s), sans modifier le code.
# Il est possible d'essayer de nouvelles photos après chaque exécution de cette cellule.
def predire_image_externe(image_rgb, nom_image):
    if image_rgb is None:
        raise ValueError(f'Image non lisible : {nom_image}')

    # Même traitement de taille + mêmes 52 descripteurs que pendant l'entraînement.
    image_adaptee = adapter_taille(image_rgb, TAILLE_REFERENCE)
    X_nouveau = extract_one(image_adaptee).reshape(1, -1).astype(np.float32)
    assert X_nouveau.shape == (1, 52)

    print('\n' + '=' * 60)
    print('IMAGE CHOISIE :', nom_image)
    print('Dimensions originales (H, L) :', image_rgb.shape[:2])
    print('Dimensions utilisées (L, H) :', TAILLE_REFERENCE)
    print('Vecteur de caractéristiques :', X_nouveau.shape)

    resultat_rbf = int(models['SVM RBF'].predict(X_nouveau)[0])
    meilleur_nom = comparison[0][0]
    meilleur_resultat = int(models[meilleur_nom].predict(X_nouveau)[0])

    plt.figure(figsize=(6, 5))
    plt.imshow(image_rgb)
    plt.title(
        f'Image importée : {nom_image}\n'
        f'SVM RBF : {CLASS_NAMES[resultat_rbf]} | '
        f'Meilleur du test ({meilleur_nom}) : {CLASS_NAMES[meilleur_resultat]}'
    )
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print('Prédictions des SIX modèles :')
    for nom_modele, modele in models.items():
        resultat = int(modele.predict(X_nouveau)[0])
        print(f'  {nom_modele:20s} → {CLASS_NAMES[resultat]}')

    print('Important : une photo sans vraie étiquette ne permet pas de calculer son accuracy.')
    print('Le modèle ne distingue que Voiture et Bateau.')


if EN_COLAB:
    print('\nCHOISIS UNE OU PLUSIEURS IMAGES DEPUIS TON ORDINATEUR :')
    fichiers_ajoutes = files.upload()
    if not fichiers_ajoutes:
        print('Aucune image importée. Relance cette cellule pour en choisir une.')
    for nom, contenu in fichiers_ajoutes.items():
        if Path(nom).suffix.lower() not in IMAGE_EXTENSIONS:
            print('Fichier ignoré (format non pris en charge) :', nom)
            continue
        image_bgr = cv2.imdecode(np.frombuffer(contenu, dtype=np.uint8), cv2.IMREAD_COLOR)
        if image_bgr is None:
            print('Image illisible :', nom)
            continue
        predire_image_externe(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB), nom)
else:
    # Utilisation en local (VS Code/terminal) : aucun package google.colab requis.
    chemin_saisi = input('\nEntre le chemin de ta nouvelle image (Entrée pour ignorer) : ').strip().strip('"')
    if chemin_saisi:
        predire_image_externe(charger_image(chemin_saisi), Path(chemin_saisi).name)
